# Notes

- made github repo
- created gitignore file
- created new venv ```python -m venv gym-dashboard```
- activated venv doing ```source gym-dashboard/Scripts/activate```
    - did source and not without it as im using git bash terminal
- install relevant libraies needed (for now) using pip install
- generated requirments txt using `pip freeze > requirements.txt`

#  Stage 1 Data Exploration

In [30]:
import pandas as pd
import os 
import math
import random


workouts = pd.read_csv('../data/strong_workouts.csv')

# print(workouts.to_string())

print(workouts.head(10))
print(workouts.tail(10))




                  Date Workout Name Duration         Exercise Name Set Order  \
0  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row         1   
1  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row         2   
2  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row         3   
3  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row         4   
4  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row         5   
5  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)         1   
6  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)         2   
7  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)         3   
8  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)         4   
9  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)         5   

   Weight  Reps  Distance  Seconds Notes  \
0   10.00  10.0       0.0      0.0   NaN   
1   10.00  10.0       0.0      

## Structure
- Columns
    - Date
    - Workout Name
    - Duration 
    - Exercise Name
    - Set (Type of set, to failure, dropset, rest, etc,)
    - Set (set number)
    - Weight (double / float in kilos 2 sf)
    - Reps (double / float to 1sf )
    - Distance
    - Seconds (how long i rested)
    - Notes
    - Workout Notes
    - RPE (Rate of Perceived Exertion 1-10)

    UPDATE: I did not know there was a info() method for pandas. i have used it below
    

- Each row is a set of an exericse in a workout with number of reps, weight, etc
-  The info method has told us the non-null values. looks like that Notes, Workout Notes RPE have the most null values (least non-null values)
    - we need to make some replacements for cleaner display when using power BI later on
    - Notes (Null) - replace null values with no notes avalaible for this set
    - Workout Notes - replace null values with no workout notes available for this set
    - RPE - ive never used this in the app so either this can be removed or somehow calculated, would be cool to see how we can calculate this, based on previous sessions 
    

In [31]:
print(workouts.info())

<class 'pandas.DataFrame'>
RangeIndex: 9038 entries, 0 to 9037
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           9038 non-null   str    
 1   Workout Name   9038 non-null   str    
 2   Duration       9038 non-null   str    
 3   Exercise Name  9038 non-null   str    
 4   Set Order      9038 non-null   str    
 5   Weight         9038 non-null   float64
 6   Reps           9038 non-null   float64
 7   Distance       9038 non-null   float64
 8   Seconds        9038 non-null   float64
 9   Notes          138 non-null    str    
 10  Workout Notes  18 non-null     str    
 11  RPE            1 non-null      float64
dtypes: float64(5), str(7)
memory usage: 847.4 KB
None


## Data Exploration II

- got some feedback from GPT to see what else I could explore
    - duplicates
    - consistent names for columns 
    - values that seem suspicious or impossible 
    - columns that are not useful




In [32]:
workout_column = (workouts['Workout Name']).unique()
print(workout_column.tolist())

duration_column = (workouts['Duration']).unique()
print(duration_column.tolist())

exercise_column = (workouts['Exercise Name']).unique()
print(exercise_column.tolist())

distance_column = (workouts['Distance']).unique()
print(distance_column.tolist())

notes_column = (workouts['Notes']).unique()
print(notes_column.tolist())

workout_notes_column = (workouts['Workout Notes']).unique()
print(workout_notes_column.tolist())

rpe_column = (workouts['RPE']).unique()
print(rpe_column.tolist())




['PULL DAY', 'PUSH DAY', 'PULL 1', 'PUSH 1', 'Legs', 'Afternoon Workout', 'Midday Workout', 'Pull day (Home)', 'Pull day HOME', 'Evening Workout', 'Morning Workout', 'HOMEPUSH', 'PUSH', '🤒 Push Day Buu', 'Afternoon Or Workout', 'VALE FARM INFLATED WEIGHTS', 'Legs + Shoulders', 'pull (weak Sesh) I Lied', 'Early Morning Workout', 'Light Sesh Due To Lack Of Sleep', 'Midnight Workout', '2 Hour Sleep Workout', '5 Hr Sleep And ILL', 'PUSH (nosleep:(', 'Leg Day (first One In A While)', 'Sarms', 'Chest + Arms', 'New Template']
['55m', '1h 25m', '52m', '1h 21m', '1h 33m', '1h 47m', '1h 45m', '1h 15m', '51m', '1h 10m', '1h 52m', '1h 23m', '1h 30m', '56m', '1h 4m', '1h 3m', '1h 7m', '1h 5m', '1h 28m', '58m', '1h 6m', '31m', '1h 8m', '1h 39m', '1h 27m', '1h 16m', '1h 11m', '1h 59m', '1h 41m', '1h 55m', '47m', '2h 1m', '1h 38m', '1h 46m', '41m', '1h 29m', '1h 12m', '2h 4m', '1h 40m', '1h 18m', '1h 24m', '32m', '1h 31m', '1h 20m', '1h 17m', '1h 37m', '50m', '48m', '54m', '57m', '21m', '1h 9m', '44m'

Exercise Volume = Weight × Reps

Workout Volume = sum(all exercise volume in workout)

Muscle Group Volume = sum(volume by category)

Workout Name: Evening Workout
Workout Category: Unknown
Muscle Group: ?

## Data Exploration Findings
- There are 11 columns as listed above
- Each entry (row) is a set in a workout
- RPE can be removed as a column as it has only 1 non-null value
- Duration: some workouts have very long durations, anything above 3 hours should be reduced down to average OR 2 hours
- Exercise Name, this is fine, there are custom names but exercises are releatively the same, there is no category for exericses, going have to do this manually probably
- Set Order - this is fine, just dictates the order in which sets were performed for an exercise
- Notes, this can be left
- workout notes, should be removed, only 18 for exericses
- Distance - bit of a strange column, majority of exercises, this value will be 0, only for cardio related exercises have actual values
- Date - important and needed to group rows for a SINGLE workout, as one row is one set for a specific exercises
- workout name, some names are not meaningful, (such as evening workout), i would preferebly like the name to describe what muscles groups are trained in the workout






# Stage 2 Data Cleaning

Standardise everything. 
Fix naming inconsistencies, 
handle missing data, 
correct data types, 
create any calculated 
fields you'll need.

In [38]:
newdf = workouts.drop("RPE", axis='columns')
newdf = newdf.fillna({"Notes": "N/A"})
newdf = newdf.fillna({"Workout Notes": "N/A"})

print(newdf.info())
print(newdf.head(1))

<class 'pandas.DataFrame'>
RangeIndex: 9038 entries, 0 to 9037
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           9038 non-null   str    
 1   Workout Name   9038 non-null   str    
 2   Duration       9038 non-null   str    
 3   Exercise Name  9038 non-null   str    
 4   Set Order      9038 non-null   str    
 5   Weight         9038 non-null   float64
 6   Reps           9038 non-null   float64
 7   Distance       9038 non-null   float64
 8   Seconds        9038 non-null   float64
 9   Notes          9038 non-null   str    
 10  Workout Notes  9038 non-null   str    
dtypes: float64(4), str(7)
memory usage: 776.8 KB
None
                  Date Workout Name Duration Exercise Name Set Order  Weight  \
0  2021-08-11 15:10:03     PULL DAY      55m     T Bar Row         1    10.0   

   Reps  Distance  Seconds Notes  \
0  10.0       0.0      0.0   N/A   

                                       Work

In [39]:
# Workout Duration
# format is xh ym where x is hours and y is minutes.
# there are many rows where i forget to stop the clock and causes my workout to run for a long time
# fix this by replacing all those values with a random number between 90 and 120 (1.5hrs and 2hrs), thats typically how long my workouts last and duration isnt in my interest
# not sure how i can make the duration part more accurate, mean?, median?
# next issue is coverting the xh ym format into an actual integer 

def duration_coversion(duration):
    hour_index = duration.find("h")
    minute_index = duration.find("m")
    hours = 0
    minutes = 0
    total_time = 0
    if (hour_index != -1 and minute_index != -1):

        hours = (int(duration[0:hour_index])) * 60

        duration_without_hours = duration[hour_index+1:len(duration)+1]
        minute_index = duration_without_hours.find("m")
        minutes = (int(duration_without_hours[0:minute_index]))

        total_time = hours + minutes

    elif hour_index != -1:
        hours = (int(duration[0:hour_index])) * 60
        total_time = hours + minutes
    else:
        minutes = (int(duration[0:minute_index]))
        total_time = hours + minutes

    return total_time

def integer_to_duration_coversion(duration):
    mins = str(duration % 60)
    hours = str(duration // 60)

    if mins == "0":
        duration_string = hours + "h"
    elif hours == "0":
        duration_string = mins + "m"
    else:
        duration_string = hours + "h" + " " + mins + "m"

    return duration_string


max_duration = duration_coversion("3h")

for x in newdf.index:
    current_duration = duration_coversion(newdf.loc[x,"Duration"])
    current_date = newdf.loc[x,"Date"]
    current_workout_name = newdf.loc[x,"Workout Name"]
    if current_duration <= max_duration:
        continue

    num = random.randint(90, 120)
    num_as_string = integer_to_duration_coversion(num)
    newdf.loc[x,"Duration"] = num_as_string
    for y in range(x, len(newdf)):
        current_date_i = newdf.loc[y,"Date"]
        current_workout_name_i = newdf.loc[y,"Workout Name"]

        if ((current_date_i == current_date) and (current_workout_name_i == current_workout_name)):
            newdf.loc[y,"Duration"] = num_as_string
        else:
            break
 
print(newdf.tail(10))


        

                     Date     Workout Name Duration           Exercise Name  \
9028  2026-06-15 20:26:31  Evening Workout   1h 58m   Reverse Fly (Machine)   
9029  2026-06-15 20:26:31  Evening Workout   1h 58m   Reverse Fly (Machine)   
9030  2026-06-15 20:26:31  Evening Workout   1h 58m   Reverse Fly (Machine)   
9031  2026-06-15 20:26:31  Evening Workout   1h 58m     Reverse curl single   
9032  2026-06-15 20:26:31  Evening Workout   1h 58m     Reverse curl single   
9033  2026-06-15 20:26:31  Evening Workout   1h 58m     Reverse curl single   
9034  2026-06-15 20:26:31  Evening Workout   1h 58m     Reverse curl single   
9035  2026-06-15 20:26:31  Evening Workout   1h 58m     Reverse curl single   
9036  2026-06-15 20:26:31  Evening Workout   1h 58m  Reverse Curl (Machine)   
9037  2026-06-15 20:26:31  Evening Workout   1h 58m  Reverse Curl (Machine)   

       Set Order  Weight  Reps  Distance  Seconds Notes Workout Notes  
9028  Rest Timer     0.0   0.0       0.0    120.0   N/A   

In [40]:
exercise_to_muscle = {
    # Chest
    "Bench Press (Barbell)": "mid_chest",
    "Bench Press (Smith Machine)": "mid_chest",
    "Bench Press (Dumbbell)": "mid_chest",
    "Chest Press (Machine)": "mid_chest",
    "Iso-Lateral Chest Press (Machine)": "mid_chest",
    "Wide Chest Press": "mid_chest",
    "Flat bench machine": "mid_chest",
    "landmine chest press": "mid_chest",

    "Incline Bench Press (Barbell)": "upper_chest",
    "Incline Bench Press (Dumbbell)": "upper_chest",
    "Incline Bench Press (Smith Machine)": "upper_chest",
    "Incline Chest Press (Machine)": "upper_chest",
    "Incline Chest Fly (Dumbbell)": "upper_chest",
    "Low to high": "upper_chest",

    "Cable Crossover": "lower_chest",
    "Chest Fly": "lower_chest",
    "Chest Fly (Band)": "lower_chest",
    "Chest Dip": "lower_chest",
    "Weighted dips": "lower_chest",
    "Dip machine": "lower_chest",
    "Push Up": "lower_chest",
    "Diamon Pushups": "lower_chest",

    # Lats
    "Lat Pulldown (Cable)": "lats",
    "Lat Pulldown (Machine)": "lats",
    "Lat Pulldown (Single Arm)": "lats",
    "Lat Pulldown - Underhand (Band)": "lats",
    "Pull Up": "lats",
    "Pull Up (Assisted)": "lats",
    "Wide Pull Up": "lats",
    "Chin Up": "lats",
    "Lat pullover": "lats",
    "Jpg lat": "lats",

    # Mid Back
    "T Bar Row": "mid_back",
    "Pendlay Row (Barbell)": "mid_back",
    "Bent Over Row (Barbell)": "mid_back",
    "Bent Over Row - Underhand (Barbell)": "mid_back",
    "Bent Over One Arm Row (Dumbbell)": "mid_back",
    "Single Arm Row": "mid_back",
    "Seated Row (Cable)": "mid_back",
    "Seated Row (Machine)": "mid_back",
    "Seated Row Single Arm": "mid_back",
    "Iso-Lateral Row (Machine)": "mid_back",
    "Seated row ub6": "mid_back",
    "Seated row gdwin": "mid_back",
    "Low Row Vale": "mid_back",

    # Rear Delts
    "Reverse Fly (Machine)": "rear_delts",
    "Reverse Fly (Cable)": "rear_delts",
    "Rear Delt ": "rear_delts",
    "Rear Delt Fly Single Arm ": "rear_delts",
    "Face Pull (Cable)": "rear_delts",
    "Y Raises": "rear_delts",

    # Front Delts
    "Front Raise (Barbell)": "front_delts",
    "Front Raise (Dumbbell)": "front_delts",
    "Front Raise (Cable)": "front_delts",
    "Front Raise (Plate)": "front_delts",

    # Side Delts
    "Lateral Raise (Cable)": "side_delts",
    "Lateral Raise (Dumbbell)": "side_delts",
    "Lateral Raise (Band)": "side_delts",
    "Lateral Raise (Machine)": "side_delts",
    "Lying Down Lateral Raises": "side_delts",

    # Compound Shoulder Presses
    "Overhead Press (Barbell)": "shoulders_compound",
    "Overhead Press (Smith Machine)": "shoulders_compound",
    "Overhead Press (Dumbbell)": "shoulders_compound",
    "Overhead Press (Cable)": "shoulders_compound",
    "Shoulder Press (Machine)": "shoulders_compound",
    "Shoulder Press (Plate Loaded)": "shoulders_compound",
    "Strict Military Press (Barbell)": "shoulders_compound",
    "Seated Overhead Press (Dumbbell)" : "shoulders_compound",

    # Biceps
    "Bicep Curl (Barbell)": "biceps",
    "Bicep Curl (Dumbbell)": "biceps",
    "Bicep Curl (Cable)": "biceps",
    "Bicep Curl (Machine)": "biceps",
    "Hammer Curl (Dumbbell)": "biceps",
    "Hammer Curl (Cable)": "biceps",
    "Incline Curl (Dumbbell)": "biceps",
    "Incline Curl Cable": "biceps",
    "Preacher Curl (Barbell)": "biceps",
    "Preacher Curl (Dumbbell)": "biceps",
    "Preacher Curl (Machine)": "biceps",
    "Concentration Curl (Dumbbell)": "biceps",
    "Bayesian": "biceps",

    # Brachialis / Brachioradialis
    "Reverse Curl (Barbell)": "brachialis_brachioradialis",
    "Reverse Curl (Dumbbell)": "brachialis_brachioradialis",
    "Reverse Curl (Machine)": "brachialis_brachioradialis",
    "Reverse Curl Cable": "brachialis_brachioradialis",
    "Reverse curl ": "brachialis_brachioradialis",
    "Reverse curl single": "brachialis_brachioradialis",

    # Triceps
    "Triceps Pushdown (Cable - Straight Bar)": "triceps",
    "Triceps Extension": "triceps",
    "Triceps Extension (Cable)": "triceps",
    "Triceps Extension (Dumbbell)": "triceps",
    "Katana tricep": "triceps",
    "Skullcrusher (Barbell)": "triceps",
    "Skullcrusher (Dumbbell)": "triceps",
    "JM PRESS": "triceps",
    "Seated Dip": "triceps",
    "Triceps Dip": "triceps",
    "Cross Body Extendion": "triceps",

    # Forearms
    "Seated Palms Up Wrist Curl (Dumbbell)": "forearms",
    "Wrist Curl Cable": "forearms",
    "Wrist curl barbell": "forearms",
    "Behind Wrist Curls": "forearms",
    "Wrist extension": "forearms",
    "Forearm Thingy": "forearms",
    "Sam Sulek": "forearms",

    # Traps
    "Shrug (Dumbbell)": "traps",
    "Shrug (Machine)": "traps",

    # Quads
    "Squat (Barbell)": "quads",
    "Squat (Smith Machine)": "quads",
    "Front Squat (Barbell)": "quads",
    "Hack Squat": "quads",
    "Pendulum squat": "quads",
    "Leg Press": "quads",
    "Leg Extension (Machine)": "quads",

    # Hamstrings
    "Romanian Deadlift (Barbell)": "hamstrings",
    "Stiff Leg Deadlift (Barbell)": "hamstrings",
    "Lying Leg Curl (Machine)": "hamstrings",
    "Seated Leg Curl (Machine)": "hamstrings",

    # Glutes
    "Hip Thrust (Barbell)": "glutes",
    "Hip thrust machine": "glutes",
    "Hip Abductor (Machine)": "glutes",
    "Sumo Deadlift (Barbell)": "glutes",

    # Adductors
    "Hip Adductor (Machine)": "adductors",

    # Calves
    "Standing Calf Raise (Machine)": "calves",
    "Seated Calf Raise (Machine)": "calves",
    "Seated Calf Raise (Plate Loaded)": "calves",
    "Calf Press on Leg Press": "calves",
    "Calf Press on Seated Leg Press": "calves",

    # Lower Back
    "Deadlift (Barbell)": "lower_back",
    "Back Extension": "lower_back",
    "Back Extension (Machine)": "lower_back",

    # Abs
    "Crunch (Machine)": "abs",
    "Leg raises": "abs",

    # Cardio
    "Running (Treadmill)": "cardio",
    "Walking": "cardio",
    "Battle Ropes": "cardio",
    "Jump Rope": "cardio",
    "Skipping": "cardio",
    "Stair": "cardio",
    "Skiing": "cardio",
    "Rowing (Machine)": "cardio",
    "Deadhang": "cardio"
}

- above we have all exercises and there respective muscle group
1) Make new column called "Muscle Group"
    - this tells us which muscle group is hit for each row
    - loop through each row in the csv, extract the exercise name for that entry, then in the column for muscle group, add in the muscle group from the dict
    - once this is done, we know which muscle group each row hits
2) Grouping rows together to represent a workout?
3) Make another new column called Volume. For each row, just calculate the number of reps multiplied by the weight of the exercise 
4) Later on when rows are grouped together to form a workout, the total volume for a workout can be calculated.




In [41]:
def add_muscle_groups(data, exercise_map):
    for x in data.index:
        exercise_name = data.loc[x, "Exercise Name"]

        muscle_group = "unknown"
        if exercise_name in exercise_map:
            muscle_group = exercise_map[exercise_name]

        data.loc[x, "Muscle Group"] = muscle_group

    return data


def calculate_volume(data):
    for x in data.index:
        weight = data.loc[x, "Weight"]
        reps = data.loc[x, "Reps"]

        volume = weight * reps
        data.loc[x, "Volume"] = volume

    return data


# Add the new columns
add_muscle_groups(newdf, exercise_to_muscle)
calculate_volume(newdf)

# Move the columns
cols = newdf.columns.tolist()

cols.remove("Muscle Group")
cols.remove("Volume")

cols.insert(cols.index("Exercise Name") + 1, "Muscle Group")
cols.insert(cols.index("Reps") + 1, "Volume")

newdf = newdf[cols]

print(newdf.head(10))
print(newdf.tail(10))

                  Date Workout Name Duration         Exercise Name  \
0  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row   
1  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row   
2  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row   
3  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row   
4  2021-08-11 15:10:03     PULL DAY      55m             T Bar Row   
5  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)   
6  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)   
7  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)   
8  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)   
9  2021-08-11 15:10:03     PULL DAY      55m  Lat Pulldown (Cable)   

  Muscle Group Set Order  Weight  Reps  Volume  Distance  Seconds Notes  \
0     mid_back         1   10.00  10.0  100.00       0.0      0.0   N/A   
1     mid_back         2   10.00  10.0  100.00       0.0      0.0   N/A   
2   

In [42]:
newdf.to_csv("../data/cleaned_workouts.csv", index=False)